In [ ]:
# Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.utils import shuffle

from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5
N_COMPONENTS = 10

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_PATH = DATA_DIR / "05_text_speech_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"


In [ ]:
def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def get_metrics(y_true, y_pred, y_prob):
    return {
        "WAcc": accuracy_score(y_true, y_pred),
        "UAcc": balanced_accuracy_score(y_true, y_pred),
        "auc": roc_auc_score(y_true, y_prob),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
    }


def subject_level_predictions(pred_conv):
    """Agrega conversaciones por sujeto mediante la media de probabilidad DS."""
    pred_subject = (
        pred_conv
        .groupby(["subject_id", "label", "outer_fold"], as_index=False)["prob_1"]
        .mean()
    )
    pred_subject["pred"] = (pred_subject["prob_1"] >= 0.5).astype(int)
    return pred_subject


def summarize_mean_std(df, cols):
    return pd.concat([
        df[cols].mean().round(3).rename("mean"),
        df[cols].std().round(3).rename("std"),
    ], axis=1)


def make_pipeline(text_cols, speech_cols, eeg_cols):
    """PCA se ajusta dentro del pipeline: no usa datos de test."""
    preprocess = ColumnTransformer(
        transformers=[
            ("text_pca", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=N_COMPONENTS, random_state=SEED)),
            ]), text_cols),
            ("speech_pca", Pipeline([
                ("scaler", StandardScaler()),
                ("pca", PCA(n_components=N_COMPONENTS, random_state=SEED)),
            ]), speech_cols),
            ("eeg", StandardScaler(), eeg_cols),
        ],
        remainder="drop",
    )

    return Pipeline([
        ("prep", preprocess),
        ("model", XGBClassifier()),
    ])


In [ ]:
data = pd.read_csv(INPUT_PATH)
partitions = load_partitions()

# Identificacion de modalidades.
text_cols = sorted([c for c in data.columns if c.startswith("text_")], key=lambda c: int(c.split("_", 1)[1]))
speech_cols = sorted([c for c in data.columns if c.startswith("speech_")], key=lambda c: int(c.split("_", 1)[1]))
meta_cols = {"subject_id", "avatar", "label"}
eeg_cols = [c for c in data.columns if c not in meta_cols and c not in text_cols and c not in speech_cols]
feature_cols = text_cols + speech_cols + eeg_cols

# Añadir outer_fold y conservar solo conversaciones con las tres modalidades completas.
data["subject_id"] = data["subject_id"].astype(str).str.strip().str.upper()
data["avatar"] = data["avatar"].astype(str).str.strip()
partitions = shuffle(partitions, random_state=SEED).reset_index(drop=True)

df = data.merge(partitions, on=["subject_id", "avatar"], how="inner")

n_before = len(df)
df = df.dropna(subset=feature_cols).copy()
n_removed = n_before - len(df)

print("Filas iniciales con partición:", n_before)
print("Filas eliminadas por falta de alguna modalidad:", n_removed)
print("Filas trimodales finales:", len(df))
print("Sujetos finales:", df["subject_id"].nunique())
print("Text features originales:", len(text_cols), "-> PCA", N_COMPONENTS)
print("Speech features originales:", len(speech_cols), "-> PCA", N_COMPONENTS)
print("EEG features:", len(eeg_cols))

print("\nSujetos por outer fold:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
display(pd.crosstab(df.drop_duplicates("subject_id")["outer_fold"], df.drop_duplicates("subject_id")["label"]))


Filas iniciales con partición: 600
Filas eliminadas por falta de alguna modalidad: 42
Filas trimodales finales: 558
Sujetos finales: 94
Text features originales: 768 -> PCA 10
Speech features originales: 1024 -> PCA 10
EEG features: 27

Sujetos por outer fold:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8


In [8]:
param_grid = {
    "model__max_depth": list(range(3, 12)),
    "model__n_estimators": [25, 50, 100, 200],
}

scoring = {
    "WAcc": "accuracy",
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

metric_cols = ["WAcc", "UAcc", "auc", "f1", "precision", "recall", "kappa"]


In [9]:
OUT_DIR = DATA_DIR / "07_results_text_speech_eeg_emotion_wise_PCA10"
OUT_DIR.mkdir(parents=True, exist_ok=True)

emotion_metrics_rows = []
subject_metrics_rows = []
best_params_rows = []
all_emotion_predictions = []

emotions = sorted(df["avatar"].unique())

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")
    fold_predictions = []

    for emotion in emotions:
        emo_df = df[df["avatar"] == emotion].copy()
        dev = emo_df[emo_df["outer_fold"] != fold].reset_index(drop=True)
        test = emo_df[emo_df["outer_fold"] == fold].reset_index(drop=True)

        if len(test) == 0 or dev["label"].nunique() < 2 or test["label"].nunique() < 2:
            print(f"{emotion}: omitido en fold {fold} por falta de datos/clases")
            continue

        X_dev = dev[feature_cols]
        y_dev = dev["label"].to_numpy(dtype=int)
        groups_dev = dev["subject_id"].to_numpy()

        X_test = test[feature_cols]
        y_test = test["label"].to_numpy(dtype=int)

        inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

        grid = GridSearchCV(
            estimator=make_pipeline(text_cols, speech_cols, eeg_cols),
            param_grid=param_grid,
            scoring=scoring,
            refit="UAcc",
            cv=inner_cv,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(X_dev, y_dev, groups=groups_dev)

        prob_1 = grid.best_estimator_.predict_proba(X_test)[:, 1]
        pred = (prob_1 >= 0.5).astype(int)

        pred_emo = test[["subject_id", "avatar", "label", "outer_fold"]].copy()
        pred_emo["prob_1"] = prob_1
        pred_emo["pred"] = pred
        fold_predictions.append(pred_emo)
        all_emotion_predictions.append(pred_emo)

        best_idx = grid.best_index_
        emo_metrics = get_metrics(y_test, pred, prob_1)
        emo_metrics["outer_fold"] = fold
        emo_metrics["avatar"] = emotion
        emo_metrics["cv_f1"] = grid.cv_results_["mean_test_f1"][best_idx]
        emotion_metrics_rows.append(emo_metrics)

        best_params_rows.append({
            "outer_fold": fold,
            "avatar": emotion,
            "best_max_depth": grid.best_params_["model__max_depth"],
            "best_n_estimators": grid.best_params_["model__n_estimators"],
            "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
            "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
        })

        print(f"{emotion}: CV F1={grid.cv_results_['mean_test_f1'][best_idx]:.3f} | Test F1={emo_metrics['f1']:.3f}")

    fold_pred = pd.concat(fold_predictions, ignore_index=True)
    pred_subject = subject_level_predictions(fold_pred)
    subject_metrics = get_metrics(pred_subject["label"], pred_subject["pred"], pred_subject["prob_1"])
    subject_metrics["outer_fold"] = fold
    subject_metrics["n_subjects"] = pred_subject["subject_id"].nunique()
    subject_metrics_rows.append(subject_metrics)

    print("Subject-level Test F1 agregado:", round(subject_metrics["f1"], 3))

emotion_metrics_df = pd.DataFrame(emotion_metrics_rows)
subject_metrics_df = pd.DataFrame(subject_metrics_rows)
best_params_df = pd.DataFrame(best_params_rows)
emotion_predictions_df = pd.concat(all_emotion_predictions, ignore_index=True)
subject_predictions_global = subject_level_predictions(emotion_predictions_df)
global_subject_metrics = get_metrics(
    subject_predictions_global["label"],
    subject_predictions_global["pred"],
    subject_predictions_global["prob_1"],
)

subject_summary = pd.DataFrame({
    "metric": ["Subject-level Test F1"],
    "mean": [subject_metrics_df["f1"].mean()],
    "std": [subject_metrics_df["f1"].std()],
}).round(3)

emotion_summary = (
    emotion_metrics_df
    .groupby("avatar")[["cv_f1", "f1", "UAcc", "auc"]]
    .agg(["mean", "std"])
    .round(3)
)

emotion_metrics_df.to_csv(OUT_DIR / "emotion_level_outer_metrics.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "subject_level_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "best_params_by_outer_fold_and_emotion.csv", index=False)
emotion_predictions_df.to_csv(OUT_DIR / "emotion_predictions.csv", index=False)
subject_predictions_global.to_csv(OUT_DIR / "subject_predictions_global.csv", index=False)
subject_summary.to_csv(OUT_DIR / "main_subject_level_summary.csv", index=False)

print("\nResumen por narrativa")
display(emotion_summary)

print("\nResultado principal agregado por sujeto")
display(subject_summary)

print("\nMétricas subject-level globales")
display(pd.Series(global_subject_metrics).round(3).to_frame("global"))

print("\nArchivos guardados en:", OUT_DIR)



===== OUTER FOLD 1 =====
Angry: CV F1=0.429 | Test F1=0.625
Happy: CV F1=0.466 | Test F1=0.706
Neutral1: CV F1=0.446 | Test F1=0.571
Neutral2: CV F1=0.439 | Test F1=0.714
Relax: CV F1=0.603 | Test F1=0.308
Sad: CV F1=0.433 | Test F1=0.588
Subject-level Test F1 agregado: 0.667

===== OUTER FOLD 2 =====
Angry: CV F1=0.597 | Test F1=0.533
Happy: CV F1=0.562 | Test F1=0.500
Neutral1: CV F1=0.572 | Test F1=0.375
Neutral2: CV F1=0.651 | Test F1=0.625
Relax: CV F1=0.594 | Test F1=0.625
Sad: CV F1=0.495 | Test F1=0.429
Subject-level Test F1 agregado: 0.625

===== OUTER FOLD 3 =====
Angry: CV F1=0.564 | Test F1=0.200
Happy: CV F1=0.602 | Test F1=0.429
Neutral1: CV F1=0.518 | Test F1=0.500
Neutral2: CV F1=0.541 | Test F1=0.545
Relax: CV F1=0.533 | Test F1=0.462
Sad: CV F1=0.496 | Test F1=0.429
Subject-level Test F1 agregado: 0.5

===== OUTER FOLD 4 =====
Angry: CV F1=0.467 | Test F1=0.267
Happy: CV F1=0.548 | Test F1=0.571
Neutral1: CV F1=0.507 | Test F1=0.615
Neutral2: CV F1=0.436 | Test F1=0.

cv_f1            f1          UAcc           auc       
           mean    std   mean    std   mean    std   mean    std
avatar                                                          
Angry     0.512  0.069  0.417  0.179  0.547  0.115  0.661  0.084
Happy     0.544  0.050  0.555  0.103  0.629  0.087  0.679  0.100
Neutral1  0.523  0.052  0.446  0.181  0.560  0.129  0.655  0.185
Neutral2  0.551  0.116  0.581  0.168  0.672  0.118  0.728  0.145
Relax     0.568  0.029  0.494  0.131  0.608  0.073  0.721  0.059
Sad       0.464  0.060  0.532  0.121  0.625  0.099  0.603  0.066


Resultado principal agregado por sujeto


,metric,mean,std
0,Subject-level Test F1,0.557,0.21



Métricas subject-level globales


,global
WAcc,0.713
UAcc,0.680
auc,0.759
f1,0.585
precision,0.731
recall,0.487
kappa,0.378



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/07_results_text_speech_eeg_emotion_wise_PCA10
